# V0.2

In [13]:
!pip install -U kaggle-environments

In [14]:
%%writefile main.py
import random

def agent(obs, config):
    player = obs["player"]
    me = obs["farms"][player]
    private = obs["private"]

    fx, fy = me["farmer"]
    tile = me["tiles"][fy][fx]

    # 辞書にキーが無い場合のエラーを防ぐ
    money = me.get("money", 0)
    seeds = private.get("seeds", {})
    shed = private.get("shed", {})
    day = obs.get("day", 0)

    market = []

    # --- 市場での売買ロジック ---

    wheat_seeds = seeds.get("WHEAT", 0)

    # 種がなく、お金が10以上あれば小麦の種を買う
    if wheat_seeds == 0 and money >= 10:
        market.append(["BUY_SEED", "WHEAT", 1])

    # 小屋に小麦があればすべて売る
    wheat_in_shed = shed.get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", wheat_in_shed])



    # --- 農業と移動のアクション ---
    farmer_action = None

    # 現在のタイルが空で、種があれば植える
    if tile is None and wheat_seeds > 0:
        farmer_action = ["PLANT", "WHEAT"]

    # 現在のタイルに植物が植えられている場合の処理
    elif isinstance(tile, dict) and tile.get("kind") == "PLANT":
        crop_age = day - tile.get("planted_day", day)

        # 小麦は2日経過していれば収穫
        if crop_age >= 2:
            farmer_action = ["HARVEST"]
        # その日の水やりがまだなら水やりを実行
        elif not tile.get("watered_today", True):
            farmer_action = ["WATER"]

    # 何も農業をしないターンはランダム移動
    if farmer_action is None:
        directions = ["NORTH", "SOUTH", "EAST", "WEST"]
        farmer_action = [random.choice(directions)]

    # 最終的な行動を返す
    return {"farmer": farmer_action, "hands": [], "market": market}


Overwriting main.py


In [15]:
from zoneinfo import ZoneInfo
import datetime
from kaggle_environments import make
from IPython.display import HTML



print(datetime.datetime.now(ZoneInfo("Asia/Tokyo")))


env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)

env.run(["main.py", "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

html_output = env.render(mode="html", width=800, height=600)
HTML(html_output)


Output hidden; open in https://colab.research.google.com to view.